# CA1 LFP & Sharp-Wave Ripple Analysis Pipeline
**BMSE 20254016 Jiwon Baek — Advanced Neurophysiology Methods**

---

## What this notebook does

All data come directly from the **Allen Brain Observatory Visual Coding – Neuropixels** dataset via AllenSDK.  
No synthetic data is generated anywhere in this notebook.

**What is actually in the Allen dataset:**
- Neuropixels probes targeting visual cortex + thalamus, passing incidentally through hippocampus (CA1/CA3/DG)
- LFP band: 374 channels, 1250 Hz, re-referenced to channels outside the brain
- Spike-sorted units with quality metrics (presence ratio, ISI violations, amplitude cutoff)
- Awake, passively viewing mice — **not sleep data**

**What this pipeline can legitimately do with this data:**
1. Load and inspect CA1 LFP from sessions with hippocampal coverage
2. Apply QC filters (Siegle et al. 2021 criteria)
3. Detect sharp-wave ripples (100–250 Hz) in CA1 LFP during quiescent periods
4. Compute spike-phase locking of CA1 units to the ripple band
5. Compute current source density (CSD) to confirm CA1 pyramidal layer localization

**What this pipeline cannot do (data limitation):**
- Sleep staging (no EEG/EMG in this dataset)
- SO or spindle detection (awake recordings)
- SD vs. recovery sleep comparison
- Fiber photometry or metabolic signals

**Data source:** Siegle et al. (2021) *Nature* 592:86–92. https://doi.org/10.1038/s41586-020-03171-x


---
## 0. Environment Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.signal as signal
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

print('AllenSDK imported successfully.')

---
## 1. Connect to Allen Brain Observatory Cache

In [ ]:
# ── Set your local cache directory ──────────────────────────────────────────
# First run: data will be downloaded from the Allen servers (~GBs per session).
# Subsequent runs: data is read from local NWB files.
OUTPUT_DIR = r'C:\Users\remin\OneDrive\Courses\26-1'   # Change to your preferred path
os.makedirs(OUTPUT_DIR, exist_ok=True)

manifest_path = os.path.join(OUTPUT_DIR, 'manifest.json')
cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)

print('Cache connected.')

---
## 2. Select a Session with CA1 Coverage

Not every session has hippocampal channels.  
We filter sessions to find those where at least one probe passed through CA1.

In [ ]:
sessions = cache.get_session_table()
print(f'Total sessions in dataset: {len(sessions)}')
print(sessions.head())

In [ ]:
# Find sessions that include CA1 in their structure list
# 'ecephys_structure_acronyms' column contains a list of structures per session
ca1_sessions = sessions[
    sessions.ecephys_structure_acronyms.apply(
        lambda x: 'CA1' in x if isinstance(x, list) else False
    )
]

print(f'Sessions with CA1 coverage: {len(ca1_sessions)}')
print(ca1_sessions[['genotype', 'sex', 'age_in_days',
                     'stimulus_name', 'ecephys_structure_acronyms']].head(10))

In [ ]:
# Select the first CA1 session for analysis
# Change index to analyze a different session
SESSION_ID = ca1_sessions.index.values[0]
print(f'Loading session: {SESSION_ID}')

session = cache.get_session_data(SESSION_ID)
print(f'Session loaded. Genotype: {ca1_sessions.loc[SESSION_ID, "genotype"]}')

---
## 3. Probe and Channel Inspection

In [ ]:
# Overview of all probes in this session
print('Probes in this session:')
print(session.probes[['description', 'sampling_rate', 'lfp_sampling_rate', 'has_lfp_data']])

In [ ]:
# Map each probe to the brain structures it passes through
probe_structure_map = {
    session.probes.loc[probe_id, 'description']:
    sorted(session.channels[
        session.channels.probe_id == probe_id
    ].ecephys_structure_acronym.dropna().unique().tolist())
    for probe_id in session.probes.index.values
}

for probe_desc, structures in probe_structure_map.items():
    print(f'  {probe_desc}: {structures}')

In [ ]:
# Identify the probe that passes through CA1
ca1_probe_ids = [
    probe_id for probe_id in session.probes.index.values
    if 'CA1' in session.channels[
        session.channels.probe_id == probe_id
    ].ecephys_structure_acronym.values
    and session.probes.loc[probe_id, 'has_lfp_data']
]

print(f'Probes with CA1 LFP: {ca1_probe_ids}')
CA1_PROBE_ID = ca1_probe_ids[0]

# Get CA1 channel IDs on this probe
ca1_channels = session.channels[
    (session.channels.probe_id == CA1_PROBE_ID) &
    (session.channels.ecephys_structure_acronym == 'CA1')
]
print(f'CA1 channels on probe {CA1_PROBE_ID}: {len(ca1_channels)}')
print(ca1_channels[['probe_channel_number',
                     'probe_vertical_position',
                     'ecephys_structure_acronym']].head(10))

In [ ]:
# Visualize probe trajectory (CCF coordinates)
fig, ax = plt.subplots(figsize=(5, 8))

probe_channels = session.channels[session.channels.probe_id == CA1_PROBE_ID].copy()
probe_channels = probe_channels.dropna(subset=['probe_vertical_position',
                                                'ecephys_structure_acronym'])

structure_colors = {
    'CA1': '#e41a1c', 'CA3': '#ff7f00', 'DG': '#4daf4a',
    'SUB': '#984ea3', 'POST': '#a65628'
}

structures_on_probe = probe_channels.ecephys_structure_acronym.unique()
for struct in structures_on_probe:
    mask = probe_channels.ecephys_structure_acronym == struct
    color = structure_colors.get(struct, '#888888')
    ax.scatter(
        np.zeros(mask.sum()),
        probe_channels.loc[mask, 'probe_vertical_position'],
        color=color, label=struct, s=15, alpha=0.7
    )

ax.set_xlabel('Probe (single column)')
ax.set_ylabel('Depth along probe (µm from tip)')
ax.set_title(f'Probe {CA1_PROBE_ID} — structure coverage')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

---
## 4. Unit Quality Control

QC criteria from Siegle et al. 2021 (Allen Neuropixels white paper):
- `presence_ratio > 0.80` — unit present for >80% of session
- `isi_violations < 0.50` — <50% refractory period contamination  
- `amplitude_cutoff < 0.10` — <10% of spikes below detection threshold

In [ ]:
# Load all units (no pre-filtering)
all_units = session.units.copy()
print(f'Total units (no filter): {len(all_units)}')

# CA1 units on the target probe
ca1_units_all = all_units[
    (all_units.probe_id == CA1_PROBE_ID) &
    (all_units.ecephys_structure_acronym == 'CA1')
]
print(f'CA1 units on probe (no filter): {len(ca1_units_all)}')

In [ ]:
# Plot quality metric distributions for CA1 units
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

qc_metrics = [
    ('presence_ratio',  0.80, 'Presence Ratio',  'steelblue'),
    ('isi_violations',  0.50, 'ISI Violations',  'tomato'),
    ('amplitude_cutoff',0.10, 'Amplitude Cutoff','seagreen'),
]

for ax, (metric, threshold, label, color) in zip(axes, qc_metrics):
    data = ca1_units_all[metric].dropna()
    ax.hist(data, bins=30, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(threshold, color='black', lw=2, ls='--', label=f'threshold={threshold}')
    n_pass = (data < threshold).sum() if metric != 'presence_ratio' \
             else (data > threshold).sum()
    ax.set_title(f'{label}\n(pass: {n_pass}/{len(data)})', fontsize=9)
    ax.set_xlabel(metric)
    ax.legend(fontsize=8)

plt.suptitle('CA1 Unit Quality Metrics', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Apply QC filter
ca1_units = ca1_units_all[
    (ca1_units_all.presence_ratio   >  0.80) &
    (ca1_units_all.isi_violations   <  0.50) &
    (ca1_units_all.amplitude_cutoff <  0.10)
]

print(f'CA1 units passing QC: {len(ca1_units)} / {len(ca1_units_all)}')
print(f'Mean firing rate (QC passed): {ca1_units.firing_rate.mean():.2f} Hz')
print(ca1_units[['firing_rate', 'presence_ratio',
                  'isi_violations', 'amplitude_cutoff']].describe().round(3))

---
## 5. Load CA1 LFP

In [ ]:
# Load LFP for the CA1 probe
# Note: first run downloads the NWB file (~1–2 GB), may take several minutes
print(f'Loading LFP for probe {CA1_PROBE_ID} ...')
lfp = session.get_lfp(CA1_PROBE_ID)

print(f'LFP shape: {lfp.shape}  (time × channels)')
print(f'Duration: {float(lfp.time[-1] - lfp.time[0]):.1f} s')
print(f'Sampling rate: {1.0 / float(np.median(np.diff(lfp.time.values))):.1f} Hz')

In [ ]:
# Select the CA1 channel closest to the pyramidal layer
# Strategy: use the channel with highest ripple-band power (proxy for str. pyramidale)
ca1_channel_ids = ca1_channels.index.values

# Restrict to channels present in the LFP DataArray
available_ca1_ids = np.intersect1d(ca1_channel_ids, lfp.channel.values)
print(f'CA1 channels available in LFP: {len(available_ca1_ids)}')

# Extract 60 s of CA1 LFP to find best ripple channel
LFP_FS = float(1.0 / np.median(np.diff(lfp.time.values)))
t_start, t_end = float(lfp.time[0]) + 60, float(lfp.time[0]) + 120

lfp_ca1_segment = lfp.sel(
    time=slice(t_start, t_end),
    channel=available_ca1_ids
).values   # shape: (time, channels)

# Bandpass to ripple band and find channel with peak RMS
nyq = LFP_FS / 2
sos_rip = signal.butter(4, [100/nyq, 250/nyq], btype='band', output='sos')

rip_rms = np.array([
    np.sqrt(np.mean(signal.sosfiltfilt(sos_rip, lfp_ca1_segment[:, ch])**2))
    for ch in range(lfp_ca1_segment.shape[1])
])

best_ch_idx = np.argmax(rip_rms)
RIPPLE_CHANNEL_ID = available_ca1_ids[best_ch_idx]

print(f'Best ripple channel: ID={RIPPLE_CHANNEL_ID}  '
      f'(RMS={rip_rms[best_ch_idx]*1e6:.2f} µV)')

# Visualize ripple RMS profile across CA1 depth
fig, ax = plt.subplots(figsize=(4, 5))
depths = ca1_channels.loc[
    ca1_channels.index.isin(available_ca1_ids), 'probe_vertical_position'
].values
ax.plot(rip_rms * 1e6, depths, 'o-', color='steelblue', ms=5)
ax.axhline(
    ca1_channels.loc[RIPPLE_CHANNEL_ID, 'probe_vertical_position'],
    color='red', ls='--', label='Selected ripple channel'
)
ax.set_xlabel('Ripple-band RMS (µV)')
ax.set_ylabel('Depth along probe (µm)')
ax.set_title('Ripple power profile across CA1')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 6. Current Source Density (CSD)

Pre-computed CSD from Allen SDK confirms CA1 pyramidal layer localization.  
CSD is computed in response to a full-field flash stimulus (current sink at str. pyramidale).

In [ ]:
csd = session.get_current_source_density(CA1_PROBE_ID)
print('CSD DataArray:')
print(csd)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# ── Left: CSD heatmap ────────────────────────────────────────────────────────
ax = axes[0]
csd_vals = csd.values   # shape: (channels, time)
t_csd    = csd.time.values * 1e3   # convert to ms
vmax     = np.percentile(np.abs(csd_vals), 99)

im = ax.imshow(
    csd_vals,
    aspect='auto',
    origin='lower',
    cmap='RdBu_r',
    vmin=-vmax, vmax=vmax,
    extent=[t_csd[0], t_csd[-1], 0, csd_vals.shape[0]]
)
ax.axvline(0, color='black', lw=1, ls='--', label='Stimulus onset')
plt.colorbar(im, ax=ax, label='CSD (µA/mm³)', fraction=0.03)
ax.set_xlabel('Time from flash onset (ms)')
ax.set_ylabel('Channel index (shallow → deep)')
ax.set_title('Current Source Density\n(flash-evoked, CA1 probe)')
ax.legend(fontsize=8)

# ── Right: Mean CSD profile at peak response ─────────────────────────────────
ax2 = axes[1]
# Find time index of peak absolute CSD after stimulus onset
t_zero_idx  = np.searchsorted(t_csd, 0)
peak_t_idx  = t_zero_idx + np.argmax(
    np.abs(csd_vals[:, t_zero_idx:]).max(axis=0)
)
csd_profile = csd_vals[:, peak_t_idx]

ax2.plot(csd_profile, np.arange(len(csd_profile)),
          color='steelblue', lw=1.5)
ax2.axvline(0, color='gray', ls='--', lw=1)
ax2.set_xlabel('CSD at peak response (µA/mm³)')
ax2.set_ylabel('Channel index')
ax2.set_title('CSD depth profile at peak')

plt.suptitle('CSD — CA1 pyramidal layer localization', fontsize=11)
plt.tight_layout()
plt.show()

---
## 7. Sharp-Wave Ripple Detection in CA1 LFP

Detection method (following Nitzan et al. 2022, which used this same Allen dataset):
1. Bandpass CA1 LFP to ripple band (100–250 Hz)
2. RMS envelope with 10 ms smoothing window
3. Threshold: mean + 3 SD of envelope
4. Duration criterion: 25–150 ms

**Important caveat:** these are awake SWRs occurring during quiescent periods between visual stimuli, not sleep SWRs. The detection method is the same; the physiological context differs.

In [ ]:
# Load full-session LFP for ripple channel
print('Extracting ripple channel LFP ...')
ca1_lfp_full = lfp.sel(channel=RIPPLE_CHANNEL_ID, method='nearest').values
t_full       = lfp.time.values
print(f'LFP extracted: {len(ca1_lfp_full)} samples  '
      f'({len(ca1_lfp_full)/LFP_FS/60:.1f} min)')

In [ ]:
def detect_swrs(lfp_signal, fs,
                ripple_lo=100.0, ripple_hi=250.0,
                threshold_sd=3.0,
                min_dur=0.025, max_dur=0.150):
    """
    Detect sharp-wave ripples from a single CA1 LFP channel.

    Parameters
    ----------
    lfp_signal : np.ndarray  — raw LFP (V)
    fs         : float       — sampling rate (Hz)
    ripple_lo  : float       — lower edge of ripple bandpass (Hz)
    ripple_hi  : float       — upper edge of ripple bandpass (Hz)
    threshold_sd : float     — envelope threshold (multiples of SD above mean)
    min_dur    : float       — minimum SWR duration (s)
    max_dur    : float       — maximum SWR duration (s)

    Returns
    -------
    swrs         : pd.DataFrame  — detected events
    rip_filtered : np.ndarray    — ripple-band signal
    envelope     : np.ndarray    — RMS envelope
    threshold    : float         — detection threshold value
    """
    nyq = fs / 2
    sos = signal.butter(4, [ripple_lo/nyq, ripple_hi/nyq],
                        btype='band', output='sos')
    rip_filtered = signal.sosfiltfilt(sos, lfp_signal)

    # RMS envelope (10 ms window)
    win = int(0.010 * fs)
    envelope = np.sqrt(
        np.convolve(rip_filtered**2, np.ones(win)/win, mode='same')
    )

    threshold = np.mean(envelope) + threshold_sd * np.std(envelope)
    above     = envelope > threshold

    events = []
    in_swr, swr_start = False, 0
    for i, val in enumerate(above):
        if val and not in_swr:
            in_swr    = True
            swr_start = i
        elif not val and in_swr:
            in_swr = False
            dur    = (i - swr_start) / fs
            if min_dur <= dur <= max_dur:
                seg  = rip_filtered[swr_start:i]
                pk   = swr_start + np.argmax(np.abs(seg))
                events.append({
                    'onset_idx':      swr_start,
                    'offset_idx':     i,
                    'peak_idx':       pk,
                    'onset_s':        swr_start / fs,
                    'offset_s':       i / fs,
                    'peak_s':         pk / fs,
                    'duration_ms':    dur * 1000,
                    'peak_amplitude_uv': float(np.max(np.abs(seg)) * 1e6)
                })

    return pd.DataFrame(events), rip_filtered, envelope, threshold


swrs, rip_filt, envelope, swr_thresh = detect_swrs(ca1_lfp_full, LFP_FS)

session_dur_min = len(ca1_lfp_full) / LFP_FS / 60
print(f'SWRs detected:  {len(swrs)}')
print(f'Rate:           {len(swrs)/session_dur_min:.2f} / min')
print(f'Duration:       {swrs.duration_ms.mean():.1f} ± {swrs.duration_ms.std():.1f} ms')
print(f'Peak amplitude: {swrs.peak_amplitude_uv.mean():.1f} ± '
      f'{swrs.peak_amplitude_uv.std():.1f} µV')

In [ ]:
# ── Visualize: 5 s of raw + ripple band + envelope + detected SWRs ───────────
seg_start_s = swrs.onset_s.iloc[5] - 1.0   # 1 s before 6th SWR
seg_end_s   = seg_start_s + 5.0

mask = (t_full >= seg_start_s) & (t_full <= seg_end_s)
t_seg        = t_full[mask] - seg_start_s
raw_seg      = ca1_lfp_full[mask] * 1e6
rip_seg      = rip_filt[mask]     * 1e6
env_seg      = envelope[mask]     * 1e6
thresh_uv    = swr_thresh          * 1e6

fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)

axes[0].plot(t_seg, raw_seg, color='#333333', lw=0.5)
axes[0].set_ylabel('Raw CA1 LFP (µV)')
axes[0].set_title('CA1 LFP — SWR detection example (5 s segment)')

axes[1].plot(t_seg, rip_seg, color='steelblue', lw=0.6)
axes[1].set_ylabel('Ripple band (µV)')

axes[2].plot(t_seg, env_seg, color='darkorange', lw=1.0, label='RMS envelope')
axes[2].axhline(thresh_uv, color='red', ls='--', lw=1.2,
                 label=f'Threshold ({swr_thresh*1e6:.1f} µV)')

# Shade detected SWRs in segment
for _, swr in swrs.iterrows():
    on = swr.onset_s - seg_start_s
    off = swr.offset_s - seg_start_s
    if -0.1 < on < 5.5:
        for ax in axes:
            ax.axvspan(max(on, 0), min(off, 5.0),
                       alpha=0.2, color='red', lw=0)

axes[2].set_ylabel('Envelope (µV)')
axes[2].set_xlabel('Time (s)')
axes[2].legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

---
## 8. SWR Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

axes[0].hist(swrs.duration_ms, bins=30, color='steelblue',
              edgecolor='white', alpha=0.8)
axes[0].axvline(swrs.duration_ms.mean(), color='red', ls='--', lw=1.5)
axes[0].set_xlabel('Duration (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'SWR Duration\nmean={swrs.duration_ms.mean():.1f} ms')

axes[1].hist(swrs.peak_amplitude_uv, bins=30, color='darkorange',
              edgecolor='white', alpha=0.8)
axes[1].axvline(swrs.peak_amplitude_uv.mean(), color='red', ls='--', lw=1.5)
axes[1].set_xlabel('Peak amplitude (µV)')
axes[1].set_title(f'SWR Amplitude\nmean={swrs.peak_amplitude_uv.mean():.1f} µV')

# Inter-SWR interval
isi_s = np.diff(swrs.peak_s.values)
axes[2].hist(isi_s, bins=40, color='seagreen', edgecolor='white', alpha=0.8)
axes[2].axvline(np.median(isi_s), color='red', ls='--', lw=1.5)
axes[2].set_xlabel('Inter-SWR interval (s)')
axes[2].set_title(f'Inter-SWR Interval\nmedian={np.median(isi_s):.2f} s')

plt.suptitle(f'SWR Feature Distributions  (n={len(swrs)})', fontsize=11)
plt.tight_layout()
plt.show()

---
## 9. Spike–Phase Locking of CA1 Units to Ripple Band

For each QC-passing CA1 unit, we compute the **mean vector length (MVL)** of spike phases  
within the ripple band during detected SWR events.  
High MVL = spikes are consistently timed relative to the ripple oscillation.

In [ ]:
# Instantaneous ripple phase via Hilbert transform
rip_phase = np.angle(signal.hilbert(rip_filt))

# Window around each SWR for spike collection
SWR_WINDOW_S = 0.1   # ±100 ms around SWR peak

def compute_swr_phase_locking(unit_spike_times, swr_df,
                               phase_signal, t_axis, window_s=0.1):
    """
    For a single unit, collect spike phases during SWR windows
    and compute mean vector length (MVL) and preferred phase.

    Returns dict with: n_spikes, mvl, preferred_phase_rad, rayleigh_p
    """
    spike_phases = []
    for _, swr in swr_df.iterrows():
        t_lo = swr.peak_s - window_s
        t_hi = swr.peak_s + window_s
        spikes_in_win = unit_spike_times[
            (unit_spike_times >= t_lo) & (unit_spike_times <= t_hi)
        ]
        for sp_t in spikes_in_win:
            idx = np.searchsorted(t_axis, sp_t)
            if 0 <= idx < len(phase_signal):
                spike_phases.append(phase_signal[idx])

    if len(spike_phases) < 5:
        return None

    phases = np.array(spike_phases)
    z      = np.exp(1j * phases)
    mvl    = float(np.abs(np.mean(z)))
    pref   = float(np.angle(np.mean(z)))

    # Rayleigh test for circular non-uniformity
    n = len(phases)
    R = n * mvl
    rayleigh_p = float(np.exp(-R**2 / n) if n > 0 else 1.0)

    return {
        'n_spikes':           n,
        'mvl':                mvl,
        'preferred_phase':    pref,
        'rayleigh_p':         rayleigh_p
    }


print(f'Computing spike-phase locking for {len(ca1_units)} CA1 units ...')
spl_results = []

for unit_id in ca1_units.index:
    spike_times = session.spike_times.get(unit_id, np.array([]))
    result = compute_swr_phase_locking(
        spike_times, swrs, rip_phase, t_full, window_s=SWR_WINDOW_S
    )
    if result is not None:
        result['unit_id'] = unit_id
        result['firing_rate'] = ca1_units.loc[unit_id, 'firing_rate']
        spl_results.append(result)

spl_df = pd.DataFrame(spl_results)
print(f'Units with sufficient spikes during SWRs: {len(spl_df)}')
print(f'Mean MVL: {spl_df.mvl.mean():.3f} ± {spl_df.mvl.std():.3f}')
print(f'Rayleigh p < 0.05: {(spl_df.rayleigh_p < 0.05).sum()} / {len(spl_df)} units')

In [ ]:
fig = plt.figure(figsize=(14, 4))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# ── MVL distribution ──────────────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0])
ax0.hist(spl_df.mvl, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax0.axvline(spl_df.mvl.mean(), color='red', ls='--', lw=1.5,
             label=f'mean={spl_df.mvl.mean():.3f}')
ax0.set_xlabel('Mean Vector Length (MVL)')
ax0.set_ylabel('Unit count')
ax0.set_title('Spike-Ripple Phase Locking')
ax0.legend(fontsize=8)

# ── Polar histogram of preferred phases (significant units only) ──────────────
ax1 = fig.add_subplot(gs[1], projection='polar')
sig_units = spl_df[spl_df.rayleigh_p < 0.05]
if len(sig_units) > 0:
    phase_bins = np.linspace(-np.pi, np.pi, 17)
    counts, edges = np.histogram(sig_units.preferred_phase, bins=phase_bins)
    centers = (edges[:-1] + edges[1:]) / 2
    max_c = counts.max() if counts.max() > 0 else 1
    ax1.bar(centers, counts/max_c, width=float(np.diff(edges)[0]),
             alpha=0.7, color='steelblue')
ax1.set_title(f'Preferred Phase\n(Rayleigh p<0.05, n={len(sig_units)})',
               pad=15, fontsize=9)

# ── MVL vs. firing rate ───────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[2])
ax2.scatter(spl_df.firing_rate, spl_df.mvl,
             c=spl_df.rayleigh_p < 0.05,
             cmap='RdYlGn_r', alpha=0.7, s=30, edgecolors='none')
r, p = stats.spearmanr(spl_df.firing_rate, spl_df.mvl)
ax2.set_xlabel('Firing rate (Hz)')
ax2.set_ylabel('MVL')
ax2.set_title(f'Firing rate vs. Phase locking\nSpearman r={r:.2f}, p={p:.3f}')

plt.suptitle('CA1 Unit — Ripple Band Phase Locking', fontsize=11)
plt.tight_layout()
plt.show()

---
## 10. SWR-Triggered Average LFP (Ripple Waveform)

Aligns raw and ripple-band LFP to SWR peak times to show the canonical ripple waveform.

In [ ]:
PRE_S  = 0.15   # 150 ms before SWR peak
POST_S = 0.15   # 150 ms after
pre_samps  = int(PRE_S  * LFP_FS)
post_samps = int(POST_S * LFP_FS)
t_win_ms   = np.linspace(-PRE_S*1000, POST_S*1000, pre_samps + post_samps)

raw_traces, rip_traces = [], []

for _, swr in swrs.iterrows():
    pk = swr.peak_idx
    s  = pk - pre_samps
    e  = pk + post_samps
    if s < 0 or e > len(ca1_lfp_full):
        continue
    raw_traces.append(ca1_lfp_full[s:e] * 1e6)
    rip_traces.append(rip_filt[s:e]     * 1e6)

raw_traces = np.array(raw_traces)
rip_traces = np.array(rip_traces)

print(f'SWR-triggered traces: {raw_traces.shape[0]} events × {raw_traces.shape[1]} samples')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, traces, label, color in [
    (axes[0], raw_traces, 'Raw CA1 LFP', '#333333'),
    (axes[1], rip_traces, 'Ripple band',  'steelblue')
]:
    mean_tr = traces.mean(axis=0)
    sem_tr  = traces.std(axis=0) / np.sqrt(len(traces))
    ax.plot(t_win_ms, mean_tr, color=color, lw=2, label='Mean')
    ax.fill_between(t_win_ms, mean_tr - sem_tr, mean_tr + sem_tr,
                     alpha=0.25, color=color, label='±SEM')
    ax.axvline(0, color='red', ls='--', lw=1, label='SWR peak')
    ax.set_xlabel('Time from SWR peak (ms)')
    ax.set_ylabel('LFP (µV)')
    ax.set_title(f'{label} — SWR-triggered average\n(n={len(traces)} events)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 11. Export Results

In [ ]:
os.makedirs('./outputs', exist_ok=True)

# SWR event table
swrs.to_csv('./outputs/swr_events.csv', index=False)

# Spike-phase locking table
spl_df.to_csv('./outputs/spike_phase_locking_ca1.csv', index=False)

# CA1 unit quality table
ca1_units[[
    'firing_rate', 'presence_ratio', 'isi_violations',
    'amplitude_cutoff', 'probe_channel_number', 'ecephys_structure_acronym'
]].to_csv('./outputs/ca1_units_qc.csv')

# Summary stats
summary = pd.Series({
    'session_id':              SESSION_ID,
    'ca1_probe_id':            CA1_PROBE_ID,
    'ripple_channel_id':       RIPPLE_CHANNEL_ID,
    'lfp_fs_hz':               LFP_FS,
    'n_swr':                   len(swrs),
    'swr_rate_per_min':        round(len(swrs) / session_dur_min, 3),
    'swr_duration_mean_ms':    round(swrs.duration_ms.mean(), 2),
    'swr_duration_sd_ms':      round(swrs.duration_ms.std(), 2),
    'swr_amplitude_mean_uv':   round(swrs.peak_amplitude_uv.mean(), 2),
    'n_ca1_units_qc':          len(ca1_units),
    'n_units_with_spl':        len(spl_df),
    'mean_mvl':                round(spl_df.mvl.mean(), 4),
    'n_significant_units_rayleigh': int((spl_df.rayleigh_p < 0.05).sum()),
})
summary.to_csv('./outputs/summary.csv', header=False)

print('Exported:')
for f in ['swr_events.csv', 'spike_phase_locking_ca1.csv',
           'ca1_units_qc.csv', 'summary.csv']:
    print(f'  ./outputs/{f}')

---
## Notes: Limitations and Scope

| What was done | Data source | Legitimate? |
|---|---|---|
| CA1 LFP loading + channel selection | Allen Visual Coding Neuropixels | ✅ |
| Unit QC filtering | Allen quality metrics (Kilosort2 output) | ✅ |
| CSD for layer localization | Allen pre-computed (flash stimulus) | ✅ |
| SWR detection (100–250 Hz RMS threshold) | Allen CA1 LFP | ✅ |
| Spike–ripple phase locking (MVL) | Allen spike times × LFP | ✅ |
| SWR-triggered average LFP | Allen CA1 LFP | ✅ |

**What is missing vs. the proposal (requires dedicated sleep recording experiment):**

| Required measurement | Why absent |
|---|---|
| Sleep staging (NREM/REM/Wake) | No EEG/EMG in Allen dataset |
| Slow oscillation detection | Awake recordings only |
| Sleep spindle detection | No thalamocortical sleep LFP |
| SO–spindle–SWR coupling | Requires sleep + mPFC + CA1 simultaneously |
| SD vs. recovery sleep comparison | Single session per mouse, no SD manipulation |
| Fiber photometry / metabolic signals | Not in dataset |

**Reference for SWR detection method applied to this dataset:**  
Nitzan, N. et al. (2022). Propagation of hippocampal ripples to the neocortex by way of a subiculum-retrosplenial pathway. *Nature Neuroscience*, 25, 641–649.